In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym
import numpy as np

# تنظیمات دستگاه
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# شبکه Actor-Critic (دو خروجی: سیاست و ارزش)
class ActorCritic(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ActorCritic, self).__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.actor = nn.Linear(hidden_dim, output_dim)  # لجیت‌های عمل
        self.critic = nn.Linear(hidden_dim, 1)          # ارزش وضعیت

    def forward(self, x):
        shared_out = self.shared(x)
        logits = self.actor(shared_out)
        value = self.critic(shared_out)
        probs = torch.softmax(logits, dim=-1)
        return probs, value

# محیط
env = gym.make('CartPole-v1')

# پارامترها
input_dim = env.observation_space.shape[0]  # 4
output_dim = env.action_space.n             # 2
hidden_dim = 128
lr = 0.001
gamma = 0.99
num_episodes = 500

# شبکه و بهینه‌ساز
model = ActorCritic(input_dim, hidden_dim, output_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)

# آموزش Online Actor-Critic
for episode in range(num_episodes):
    state = env.reset()[0]  # در Gym v0.26+
    total_reward = 0
    done = False

    while not done:
        # تبدیل وضعیت به تانسور
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)

        # دریافت سیاست و ارزش فعلی
        probs, value = model(state_tensor)
        value_current = value.squeeze()

        # انتخاب عمل با نمونه‌گیری از توزیع
        action_dist = torch.distributions.Categorical(probs)
        action = action_dist.sample()
        log_prob = action_dist.log_prob(action)

        # انجام عمل
        next_state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        total_reward += reward

        # وضعیت بعدی
        next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0).to(device)
        with torch.no_grad():
            _, value_next = model(next_state_tensor)
            value_next = value_next.squeeze()

        # محاسبه TD Error (Advantage تخمینی)
        if done:
            td_target = reward
        else:
            td_target = reward + gamma * value_next

        td_error = td_target - value_current  # δ = r + γV(s') - V(s)

        # تابع زیان: Actor Loss + Critic Loss
        actor_loss = -log_prob * td_error.detach()  # فقط گرادیان از log_prob بگیرد
        critic_loss = td_error.pow(2)  # MSE: (TD Target - V(s))^2
        total_loss = actor_loss + 0.5 * critic_loss

        # به‌روزرسانی وزن‌ها
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        # رفتن به وضعیت بعدی
        state = next_state

    # نمایش پیشرفت
    if episode % 50 == 0:
        print(f"Episode {episode}, Total Reward: {total_reward}, "
              f"Loss: {total_loss.item():.4f}")

print("Training finished.")